# DreamerV3 Walker-Walk Demo & Smoke Test
This notebook demonstrates setup, training, and learning curve visualization on DMC `walker_walk` with proprioceptive inputs.
- Codebase: `NM512/dreamerv3-torch` (commit `6ef8646d807cd10ce0c88e10a7e943211e7fc44c`)
- Budget: 1,000 simulator frames (smoke test)

In [ ]:
# 1. Environment & Hardware Verification
import os, sys, torch
print('Python Version: ', sys.version.split()[0])
print('PyTorch Version:', torch.__version__)
print('CUDA Available: ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:    ', torch.cuda.get_device_name(0))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Active Device: ', device)

In [ ]:
# 2. Run 1,000-Step Baseline Smoke Test
import subprocess
logdir = 'smoke_log_demo'
cmd = [
    sys.executable, 'dreamer.py',
    '--configs', 'dmc_proprio',
    '--task', 'dmc_walker_walk',
    '--logdir', logdir,
    '--steps', '1000',
    '--eval_every', '500',
    '--prefill', '250',
    '--eval_episode_num', '1',
    '--seed', '0',
    '--device', device
]
print('Executing:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print('Exit code:', res.returncode)
if res.returncode != 0:
    print('STDERR:', res.stderr[-1500:])
    raise RuntimeError('Smoke test failed')
else:
    print('[SUCCESS] Smoke test completed successfully!')

In [ ]:
# 3. Parse and Plot Metrics
import json, pathlib
import matplotlib.pyplot as plt

metrics_path = pathlib.Path(logdir) / 'metrics.jsonl'
steps, returns = [], []
if metrics_path.exists():
    with open(metrics_path) as f:
        for line in f:
            d = json.loads(line)
            if 'eval_return' in d:
                steps.append(d['step'])
                returns.append(d['eval_return'])
    print(f'Found {len(returns)} evaluation checkpoints:', list(zip(steps, returns)))
    plt.figure(figsize=(6, 3.5))
    plt.plot(steps, returns, marker='o', color='royalblue', label='Evaluation Return')
    plt.xlabel('Simulator Frames')
    plt.ylabel('Episodic Return')
    plt.title('DreamerV3 Walker-Walk Smoke Test')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('[WARN] No metrics.jsonl found.')